In [9]:
import sqlite3
import joblib
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

In [10]:
conn = sqlite3.connect("../data/soccer.db")

In [11]:
scaler = StandardScaler()
ROLLING_WINDOW = 5
INITIAL_ELO = 1500
ELO_K = 20
HOME_ADVANTAGE = 100

In [12]:
query = """
SELECT
    m.*,
    c.name AS competition_name

FROM matches m

JOIN competitions c
ON m.competition_id = c.id
"""

matches_df = pd.read_sql(query, conn)

In [13]:
matches_df["result"] = matches_df.apply(
    lambda row:
        "H" if row["home_goals"] > row["away_goals"]
        else "A" if row["home_goals"] < row["away_goals"]
        else "D",
    axis=1
)
matches_df["total_goals"] = (
    matches_df["home_goals"] +
    matches_df["away_goals"]
)
matches_df["over_2_5"] = (
    matches_df["total_goals"] > 2.5
)
matches_df["btts"] = (
    (matches_df["home_goals"] > 0) &
    (matches_df["away_goals"] > 0)
)
matches_df["is_draw"] = (
    matches_df["home_goals"] ==
    matches_df["away_goals"]
)

In [14]:
completed_matches = matches_df[
    matches_df["status"].isin([
        "FT",
        "AET",
        "PEN"
    ])
].copy()

In [15]:
completed_matches = completed_matches.sort_values(
    by="date"
)

In [16]:
completed_matches = completed_matches.sort_values(by="date").copy()

completed_matches["result"] = completed_matches.apply(
    lambda row:
        "H" if row["home_goals"] > row["away_goals"]
        else "A" if row["home_goals"] < row["away_goals"]
        else "D",
    axis=1
)

completed_matches["home_win"] = (
    completed_matches["result"] == "H"
)

completed_matches["total_goals"] = (
    completed_matches["home_goals"] +
    completed_matches["away_goals"]
)

completed_matches["over_2_5"] = (
    completed_matches["total_goals"] > 2.5
)

completed_matches["btts"] = (
    (completed_matches["home_goals"] > 0) &
    (completed_matches["away_goals"] > 0)
)

completed_matches["is_draw"] = (
    completed_matches["home_goals"] ==
    completed_matches["away_goals"]
)

In [17]:
home_df = completed_matches[
    [
        "date",
        "competition_name",
        "season",
        "home_team_id",
        "home_goals",
        "away_goals"
    ]
].copy()

home_df.columns = [
    "date",
    "competition_name",
    "season",
    "team_id",
    "goals_scored",
    "goals_conceded"
]

away_df = completed_matches[
    [
        "date",
        "competition_name",
        "season",
        "away_team_id",
        "away_goals",
        "home_goals"
    ]
].copy()

away_df.columns = [
    "date",
    "competition_name",
    "season",
    "team_id",
    "goals_scored",
    "goals_conceded"
]

team_history = pd.concat([home_df, away_df])

team_history = team_history.sort_values(
    by=["team_id", "date"]
)

team_history["rolling_goals_scored"] = (
    team_history
    .groupby("team_id")["goals_scored"]
    .transform(
        lambda x: x.shift(1).rolling(ROLLING_WINDOW).mean()
    )
)

team_history["rolling_goals_conceded"] = (
    team_history
    .groupby("team_id")["goals_conceded"]
    .transform(
        lambda x: x.shift(1).rolling(ROLLING_WINDOW).mean()
    )
)

In [18]:
home_features = team_history.copy()

home_features = home_features.rename(
    columns={
        "team_id": "home_team_id",
        "rolling_goals_scored": "home_rolling_scored",
        "rolling_goals_conceded": "home_rolling_conceded"
    }
)

away_features = team_history.copy()

away_features = away_features.rename(
    columns={
        "team_id": "away_team_id",
        "rolling_goals_scored": "away_rolling_scored",
        "rolling_goals_conceded": "away_rolling_conceded"
    }
)

matches_features = completed_matches.merge(
    home_features[
        [
            "date",
            "home_team_id",
            "home_rolling_scored",
            "home_rolling_conceded"
        ]
    ],
    on=["date", "home_team_id"],
    how="left"
)

matches_features = matches_features.merge(
    away_features[
        [
            "date",
            "away_team_id",
            "away_rolling_scored",
            "away_rolling_conceded"
        ]
    ],
    on=["date", "away_team_id"],
    how="left"
)
matches_features["home_dnb"] = (
    matches_features["result"] != "A"
)
matches_features["away_dnb"] = (
    matches_features["result"] != "H"
)
matches_features["round_number"] = (

    matches_features["round"]

    .str.extract(r"(\d+)")[0]

    .astype(float)
)

In [19]:
print(matches_features.shape)

(21249, 26)


In [20]:
elo_matches = completed_matches.sort_values(by="date").copy()

elo_ratings = {}
elo_home = []
elo_away = []

K = ELO_K

for _, match in elo_matches.iterrows():

    home_team = match["home_team_id"]
    away_team = match["away_team_id"]

    home_elo = elo_ratings.get(home_team, INITIAL_ELO)
    away_elo = elo_ratings.get(away_team, INITIAL_ELO)

    elo_home.append(home_elo)
    elo_away.append(away_elo)

    expected_home = 1 / (1 + 10 ** ((away_elo - home_elo) / 400))
    expected_away = 1 - expected_home

    if match["home_goals"] > match["away_goals"]:
        actual_home = 1
        actual_away = 0
    elif match["home_goals"] < match["away_goals"]:
        actual_home = 0
        actual_away = 1
    else:
        actual_home = 0.5
        actual_away = 0.5

    new_home_elo = home_elo + K * (actual_home - expected_home)
    new_away_elo = away_elo + K * (actual_away - expected_away)

    elo_ratings[home_team] = new_home_elo
    elo_ratings[away_team] = new_away_elo

elo_matches["home_elo"] = elo_home
elo_matches["away_elo"] = elo_away

matches_features = matches_features.merge(
    elo_matches[
        [
            "id",
            "home_elo",
            "away_elo"
        ]
    ],
    on="id",
    how="left"
)

In [21]:
features = [
    "home_rolling_scored",
    "away_rolling_scored",
    "home_rolling_conceded",
    "away_rolling_conceded",
    "home_elo",
    "away_elo",
]

In [22]:
print(features)

['home_rolling_scored', 'away_rolling_scored', 'home_rolling_conceded', 'away_rolling_conceded', 'home_elo', 'away_elo']


In [23]:
matches_features.columns.tolist()

['id',
 'api_fixture_id',
 'competition_id',
 'season',
 'date',
 'round',
 'home_team_id',
 'away_team_id',
 'home_goals',
 'away_goals',
 'status',
 'referee',
 'competition_name',
 'result',
 'total_goals',
 'over_2_5',
 'btts',
 'is_draw',
 'home_win',
 'home_rolling_scored',
 'home_rolling_conceded',
 'away_rolling_scored',
 'away_rolling_conceded',
 'home_dnb',
 'away_dnb',
 'round_number',
 'home_elo',
 'away_elo']

### Correr hasta aquí

In [17]:
target = "home_win"

model_df = matches_features.dropna(
    subset=features + [target]
).copy()

train_df = model_df[
    model_df["season"] <= 2023
]

test_df = model_df[
    model_df["season"] >= 2024
]

X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)

model.fit(X_train_scaled, y_train)

predictions = model.predict(X_test_scaled)

print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

       False       0.67      0.74      0.71      2314
        True       0.62      0.54      0.58      1803

    accuracy                           0.65      4117
   macro avg       0.65      0.64      0.64      4117
weighted avg       0.65      0.65      0.65      4117



### AQUI USAREMOS XGBOOST AHORA

In [18]:
target = "home_win"

features = [
    "home_rolling_scored",
    "away_rolling_scored",

    "home_rolling_conceded",
    "away_rolling_conceded",

    "home_elo",
    "away_elo",
]

In [19]:
xgb_model = XGBClassifier(

    n_estimators=200,

    max_depth=4,

    learning_rate=0.05,

    subsample=0.8,

    colsample_bytree=0.8,

    random_state=42
)

In [20]:
xgb_model = XGBClassifier(

    n_estimators=200,

    max_depth=4,

    learning_rate=0.05,

    subsample=0.8,

    colsample_bytree=0.8,

    random_state=42
)

In [21]:
xgb_model.fit(
    X_train,
    y_train
)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_metho

In [22]:
xgb_predictions = xgb_model.predict(
    X_test
)
print(
    classification_report(
        y_test,
        xgb_predictions
    )
)

              precision    recall  f1-score   support

       False       0.68      0.73      0.70      2314
        True       0.61      0.55      0.58      1803

    accuracy                           0.65      4117
   macro avg       0.64      0.64      0.64      4117
weighted avg       0.65      0.65      0.65      4117



In [23]:
importance_df = pd.DataFrame({

    "feature": features,

    "importance": xgb_model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="importance",
    ascending=False
)

importance_df

,feature,importance
4,home_elo,0.296008
5,away_elo,0.240444
1,away_rolling_scored,0.155379
0,home_rolling_scored,0.121527
3,away_rolling_conceded,0.101487
2,home_rolling_conceded,0.085156


#### AQUI AÑADIREMOS LOS DÍAS DE DESCANSO Y DT

In [24]:
context_query = """
SELECT
    mc.match_id,

    mc.home_rest_days,
    mc.away_rest_days,

    mc.home_coach_tenure_days,
    mc.away_coach_tenure_days

FROM match_context mc
"""
context_df = pd.read_sql(
    context_query,
    conn
)

In [25]:
matches_features = matches_features.merge(

    context_df,

    left_on="id",
    right_on="match_id",

    how="left"
)

In [32]:
features = [

    "home_rolling_scored",
    "away_rolling_scored",

    "home_rolling_conceded",
    "away_rolling_conceded",

    "home_elo",
    "away_elo",

]

In [33]:
model_df[
    features
].isna().sum()

home_rolling_scored      0
away_rolling_scored      0
home_rolling_conceded    0
away_rolling_conceded    0
home_elo                 0
away_elo                 0
dtype: int64

In [34]:
target = "home_win" #over_2_5 o home_win

model_df = matches_features.dropna(
    subset=features + [target]
).copy()

train_df = model_df[
    model_df["season"] <= 2023
]

test_df = model_df[
    model_df["season"] >= 2024
]

X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)

model.fit(X_train_scaled, y_train)
probabilities = model.predict_proba(
    X_test_scaled
)
predictions = model.predict(X_test_scaled)

print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

       False       0.67      0.74      0.71      2314
        True       0.62      0.54      0.58      1803

    accuracy                           0.65      4117
   macro avg       0.65      0.64      0.64      4117
weighted avg       0.65      0.65      0.65      4117



In [29]:
model_df = matches_features.dropna(
    subset=features + [target]
).copy()

In [30]:
target = "home_dnb" #over_2_5 o home_win

features = [
    "home_rolling_scored",
    "away_rolling_scored",

    "home_rolling_conceded",
    "away_rolling_conceded",

    "home_elo",
    "away_elo",

    "home_rest_days",
    "away_rest_days",

    "home_coach_tenure_days",
    "away_coach_tenure_days"
]

In [35]:
xgb_model = XGBClassifier(

    n_estimators=200,

    max_depth=4,

    learning_rate=0.05,

    subsample=0.8,

    colsample_bytree=0.8,

    random_state=42
)
xgb_model = XGBClassifier(

    n_estimators=200,

    max_depth=4,

    learning_rate=0.05,

    subsample=0.8,

    colsample_bytree=0.8,

    random_state=42
)
xgb_model.fit(
    X_train,
    y_train
)
xgb_predictions = xgb_model.predict(
    X_test
)
print(
    classification_report(
        y_test,
        xgb_predictions
    )
)

              precision    recall  f1-score   support

       False       0.68      0.73      0.70      2314
        True       0.61      0.55      0.58      1803

    accuracy                           0.65      4117
   macro avg       0.64      0.64      0.64      4117
weighted avg       0.65      0.65      0.65      4117



HAGAMOS QUE PREDIGA LOS 3 CASOS

In [137]:
result_mapping = {

    "H": 0,
    "D": 1,
    "A": 2
}

matches_features["result_encoded"] = (
    matches_features["result"]
    .map(result_mapping)
)

In [138]:
target = "result_encoded" #over_2_5 o home_win

model_df = matches_features.dropna(
    subset=features + [target]
).copy()

train_df = model_df[
    model_df["season"] <= 2023
]

test_df = model_df[
    model_df["season"] >= 2024
]

X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000,class_weight="balanced")

model.fit(X_train_scaled, y_train)

predictions = model.predict(X_test_scaled)

print(classification_report(y_test, predictions))

ValueError: Found array with 0 sample(s) (shape=(0, 10)) while a minimum of 1 is required by StandardScaler.

In [139]:
target = "is_draw" #over_2_5 o home_win

features = [
    "home_rolling_scored",
    "away_rolling_scored",

    "home_rolling_conceded",
    "away_rolling_conceded",

    "home_elo",
    "away_elo",

    "home_rest_days",
    "away_rest_days",

    "home_coach_tenure_days",
    "away_coach_tenure_days"
]
model_df = matches_features.dropna(
    subset=features + [target]
).copy()

In [140]:
xgb_model = XGBClassifier(

    objective="multi:softprob",

    num_class=3,

    n_estimators=300,

    max_depth=6,

    learning_rate=0.03,

    subsample=0.8,

    colsample_bytree=0.8,

    random_state=42
)
xgb_model.fit(
    X_train,
    y_train
)
xgb_predictions = xgb_model.predict(
    X_test
)
print(
    classification_report(
        y_test,
        xgb_predictions
    )
)

ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:home_rest_days: object, away_rest_days: object

### Este es el importante BASADO FULL HOME NO EMPATE

In [60]:
features = [
    "home_rolling_scored",
    "away_rolling_scored",

    "home_rolling_conceded",
    "away_rolling_conceded",

    "home_elo",
    "away_elo",
    "round_number"
]

In [61]:
target = "home_dnb"

model_df = matches_features.dropna(
    subset=features + [target]
).copy()

train_df = model_df[
    model_df["season"] <= 2023
]

test_df = model_df[
    model_df["season"] >= 2024
]

X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(
    max_iter=1000
)

model.fit(
    X_train_scaled,
    y_train
)
probabilities = model.predict_proba(
    X_test_scaled
)
home_dnb_proba = probabilities[:, 1]
test_results = test_df.copy()
test_results["prediction_proba"] = (
    home_dnb_proba
)
test_results["prediction"] = (
    test_results["prediction_proba"] >= 0.75
)
confident_predictions = test_results[
    test_results["prediction_proba"] >= 0.75
]
accuracy_score(

    confident_predictions["home_dnb"],

    confident_predictions["prediction"]
)

0.8358644859813084

In [62]:
test_results["prediction"] = (
    test_results["prediction_proba"] >= 0.75
)
teams_query = """
SELECT
    id,
    name
FROM teams
"""
teams_df = pd.read_sql(
    teams_query,
    conn
)
team_map = dict(

    zip(

        teams_df["id"],

        teams_df["name"]
    )
)
test_results["home_team_name"] = (

    test_results["home_team_id"]
    .map(team_map)
)
test_results["away_team_name"] = (

    test_results["away_team_id"]
    .map(team_map)
)
test_results[
    [
        "competition_name",

        "season",

        "date",

        "home_team_name",

        "away_team_name",

        "prediction_proba",

        "home_dnb"
    ]
].sort_values(

    by="prediction_proba",

    ascending=False

).head(50)

,competition_name,season,date,home_team_name,away_team_name,prediction_proba,home_dnb
17622,Premier League,2024,2024-10-26 14:00:00.000000,Manchester City,Southampton,0.977761,True
18775,Ligue 1,2024,2025-04-05 15:00:00.000000,Paris Saint Germain,Angers,0.977328,True
18603,Premier League,2024,2025-03-08 15:00:00.000000,Liverpool,Southampton,0.976030,True
21217,Serie A,2025,2026-05-17 13:00:00.000000,Inter,Hellas Verona,0.970807,True
20585,Ligue 1,2025,2026-02-21 20:05:00.000000,Paris Saint Germain,Metz,0.970453,True
17501,Premier League,2024,2024-10-05 14:00:00.000000,Arsenal,Southampton,0.969866,True
19402,Bundesliga,2025,2025-09-13 16:30:00.000000,Bayern München,Hamburger SV,0.969337,True
19947,Bundesliga,2025,2025-11-29 14:30:00.000000,Bayern München,FC St. Pauli,0.968149,True
21085,Bundesliga,2025,2026-05-02 13:30:00.000000,Bayern München,1. FC Heidenheim,0.967408,True
18894,Ligue 1,2024,2025-04-19 15:00:00.000000,Paris Saint Germain,Le Havre,0.966375,True


### Intento de calibración

In [63]:
base_model = LogisticRegression(
    max_iter=1000
)
calibrated_model = CalibratedClassifierCV(

    base_model,

    method="sigmoid",

    cv=5
)
calibrated_model.fit(
    X_train_scaled,
    y_train
)
probabilities = calibrated_model.predict_proba(
    X_test_scaled
)
probabilities[:10]

array([[0.33201329, 0.66798671],
       [0.22005937, 0.77994063],
       [0.28307195, 0.71692805],
       [0.60988215, 0.39011785],
       [0.2499095 , 0.7500905 ],
       [0.13755456, 0.86244544],
       [0.14313581, 0.85686419],
       [0.2721503 , 0.7278497 ],
       [0.35111544, 0.64888456],
       [0.09558517, 0.90441483]])

In [64]:
home_dnb_proba = probabilities[:, 1]

test_results = test_df.copy()

test_results["prediction_proba"] = (
    home_dnb_proba
)

test_results["prediction"] = (
    test_results["prediction_proba"] >= 0.75
)

confident_predictions = test_results[
    test_results["prediction_proba"] >= 0.75
]

print(

    accuracy_score(

        confident_predictions["home_dnb"],

        confident_predictions["prediction"]
    )
)

0.8359420289855073


In [65]:
teams_query = """
SELECT
    id,
    name
FROM teams
"""
teams_df = pd.read_sql(
    teams_query,
    conn
)
team_map = dict(

    zip(

        teams_df["id"],

        teams_df["name"]
    )
)
test_results["home_team_name"] = (

    test_results["home_team_id"]
    .map(team_map)
)
test_results["away_team_name"] = (

    test_results["away_team_id"]
    .map(team_map)
)
test_results[
    [
        "competition_name",

        "season",

        "date",

        "home_team_name",

        "away_team_name",

        "prediction_proba",

        "home_dnb"
    ]
].sort_values(

    by="prediction_proba",

    ascending=False

).head(50)

,competition_name,season,date,home_team_name,away_team_name,prediction_proba,home_dnb
17622,Premier League,2024,2024-10-26 14:00:00.000000,Manchester City,Southampton,0.977737,True
18775,Ligue 1,2024,2025-04-05 15:00:00.000000,Paris Saint Germain,Angers,0.977392,True
18603,Premier League,2024,2025-03-08 15:00:00.000000,Liverpool,Southampton,0.976006,True
21217,Serie A,2025,2026-05-17 13:00:00.000000,Inter,Hellas Verona,0.970850,True
20585,Ligue 1,2025,2026-02-21 20:05:00.000000,Paris Saint Germain,Metz,0.970578,True
17501,Premier League,2024,2024-10-05 14:00:00.000000,Arsenal,Southampton,0.970005,True
19402,Bundesliga,2025,2025-09-13 16:30:00.000000,Bayern München,Hamburger SV,0.969575,True
19947,Bundesliga,2025,2025-11-29 14:30:00.000000,Bayern München,FC St. Pauli,0.968409,True
21085,Bundesliga,2025,2026-05-02 13:30:00.000000,Bayern München,1. FC Heidenheim,0.967741,True
18894,Ligue 1,2024,2025-04-19 15:00:00.000000,Paris Saint Germain,Le Havre,0.966712,True


In [47]:
from xgboost import XGBClassifier

from sklearn.metrics import (
    classification_report,
    accuracy_score
)

target = "home_dnb"

model_df = matches_features.dropna(
    subset=features + [target]
).copy()

train_df = model_df[
    model_df["season"] <= 2023
]

test_df = model_df[
    model_df["season"] >= 2024
]

X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]

xgb_model = XGBClassifier(

    n_estimators=200,

    max_depth=4,

    learning_rate=0.05,

    subsample=0.8,

    colsample_bytree=0.8,

    random_state=42
)

xgb_model.fit(
    X_train,
    y_train
)

predictions = xgb_model.predict(
    X_test
)

print(
    classification_report(
        y_test,
        predictions
    )
)

probabilities = xgb_model.predict_proba(
    X_test
)

home_dnb_proba = probabilities[:, 1]

test_results = test_df.copy()

test_results["prediction_proba"] = (
    home_dnb_proba
)

test_results["prediction"] = (
    test_results["prediction_proba"] >= 0.75
)

confident_predictions = test_results[
    test_results["prediction_proba"] >= 0.75
]

high_confidence_accuracy = accuracy_score(

    confident_predictions["home_dnb"],

    confident_predictions["prediction"]
)

print(
    f"High confidence accuracy: {high_confidence_accuracy}"
)

print(
    f"Total confident predictions: {len(confident_predictions)}"
)

              precision    recall  f1-score   support

       False       0.56      0.34      0.42      1270
        True       0.74      0.88      0.81      2782

    accuracy                           0.71      4052
   macro avg       0.65      0.61      0.61      4052
weighted avg       0.69      0.71      0.69      4052

High confidence accuracy: 0.8276440962506995
Total confident predictions: 1787


In [48]:
from xgboost import XGBClassifier

from sklearn.calibration import (
    CalibratedClassifierCV
)

from sklearn.metrics import (
    classification_report,
    accuracy_score
)

target = "home_dnb"

model_df = matches_features.dropna(
    subset=features + [target]
).copy()

train_df = model_df[
    model_df["season"] <= 2023
]

test_df = model_df[
    model_df["season"] >= 2024
]

X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]

base_model = XGBClassifier(

    n_estimators=200,

    max_depth=4,

    learning_rate=0.05,

    subsample=0.8,

    colsample_bytree=0.8,

    random_state=42
)

calibrated_model = CalibratedClassifierCV(

    base_model,

    method="sigmoid",

    cv=5
)

calibrated_model.fit(
    X_train,
    y_train
)

predictions = calibrated_model.predict(
    X_test
)

print(
    classification_report(
        y_test,
        predictions
    )
)

probabilities = calibrated_model.predict_proba(
    X_test
)

home_dnb_proba = probabilities[:, 1]

test_results = test_df.copy()

test_results["prediction_proba"] = (
    home_dnb_proba
)

test_results["prediction"] = (
    test_results["prediction_proba"] >= 0.75
)

confident_predictions = test_results[
    test_results["prediction_proba"] >= 0.75
]

high_confidence_accuracy = accuracy_score(

    confident_predictions["home_dnb"],

    confident_predictions["prediction"]
)

print(
    f"High confidence accuracy: {high_confidence_accuracy}"
)

print(
    f"Total confident predictions: {len(confident_predictions)}"
)

              precision    recall  f1-score   support

       False       0.56      0.33      0.42      1270
        True       0.74      0.88      0.81      2782

    accuracy                           0.71      4052
   macro avg       0.65      0.61      0.61      4052
weighted avg       0.68      0.71      0.68      4052

High confidence accuracy: 0.8203655352480418
Total confident predictions: 1915


In [33]:
teams_query = """
SELECT
    id,
    name
FROM teams
"""

teams_df = pd.read_sql(
    teams_query,
    conn
)

team_map = dict(

    zip(

        teams_df["id"],

        teams_df["name"]
    )
)

test_results["home_team_name"] = (

    test_results["home_team_id"]
    .map(team_map)
)

test_results["away_team_name"] = (

    test_results["away_team_id"]
    .map(team_map)
)

test_results[
    [
        "competition_name",

        "season",

        "date",

        "home_team_name",

        "away_team_name",

        "prediction_proba",

        "home_dnb"
    ]
].sort_values(

    by="prediction_proba",

    ascending=False

).head(50)

,competition_name,season,date,home_team_name,away_team_name,prediction_proba,home_dnb
18894,Ligue 1,2024,2025-04-19 15:00:00.000000,Paris Saint Germain,Le Havre,0.899403,True
19947,Bundesliga,2025,2025-11-29 14:30:00.000000,Bayern München,FC St. Pauli,0.897600,True
17913,Serie A,2024,2024-12-06 17:30:00.000000,Inter,Parma,0.896982,True
17722,Serie A,2024,2024-11-03 19:45:00.000000,Inter,Venezia,0.896929,True
17872,Ligue 1,2024,2024-11-30 20:00:00.000000,Paris Saint Germain,Nantes,0.896617,True
18603,Premier League,2024,2025-03-08 15:00:00.000000,Liverpool,Southampton,0.895774,True
21085,Bundesliga,2025,2026-05-02 13:30:00.000000,Bayern München,1. FC Heidenheim,0.895727,True
20585,Ligue 1,2025,2026-02-21 20:05:00.000000,Paris Saint Germain,Metz,0.895167,True
21246,Premier League,2025,2026-05-18 19:00:00.000000,Arsenal,Burnley,0.894978,True
18313,Bundesliga,2024,2025-02-01 14:30:00.000000,Bayern München,Holstein Kiel,0.894784,True


In [143]:
threshold = 0.80
confident_predictions = test_results[
    test_results["prediction_proba"] >= threshold
]

acc = accuracy_score(

    confident_predictions["home_dnb"],

    confident_predictions["prediction"]
)

count = len(confident_predictions)

print(
    threshold,
    acc,
    count
)

NameError: name 'test_results' is not defined

In [144]:
confident_predictions[
    "competition_name"
].value_counts()

NameError: name 'confident_predictions' is not defined

In [145]:
confident_predictions.groupby(
    "season"
)["home_dnb"].mean()

NameError: name 'confident_predictions' is not defined

In [146]:
teams_query = """
SELECT
    id,
    name
FROM teams
"""
teams_df = pd.read_sql(
    teams_query,
    conn
)
team_map = dict(
    zip(
        teams_df["id"],
        teams_df["name"]
    )
)
test_results.sort_values(
    by="prediction_proba",
    ascending=False
)[
    [
        "competition_name",
        "date",
        "home_team_id",
        "away_team_id",
        "prediction_proba",
        "home_dnb"
    ]
].head(20)
test_results["home_team_name"] = (
    test_results["home_team_id"]
    .map(team_map)
)

test_results["away_team_name"] = (
    test_results["away_team_id"]
    .map(team_map)
)
test_results.sort_values(

    by="prediction_proba",

    ascending=False

)[
    [
        "competition_name",

        "season",

        "date",

        "home_team_name",

        "away_team_name",

        "prediction_proba",

        "home_dnb"
    ]
].head(50)

NameError: name 'test_results' is not defined

In [147]:
teams_query = """
SELECT
    id,
    name
FROM teams
"""
teams_df = pd.read_sql(
    teams_query,
    conn
)
team_map = dict(
    zip(
        teams_df["id"],
        teams_df["name"]
    )
)
wrong_predictions = confident_predictions[

    confident_predictions["home_dnb"]
    !=
    confident_predictions["prediction"]
]
wrong_predictions["home_team_name"] = (
    wrong_predictions["home_team_id"]
    .map(team_map)
)

wrong_predictions["away_team_name"] = (
    wrong_predictions["away_team_id"]
    .map(team_map)
)
wrong_predictions.sort_values(

    by="prediction_proba",

    ascending=False

)[
    [
        "competition_name",

        "date",

        "home_team_name",

        "away_team_name",

        "prediction_proba",

        "home_dnb"
    ]
].head(20)

NameError: name 'confident_predictions' is not defined

In [148]:
medium_confidence = test_results[

    (
        test_results["prediction_proba"] >= 0.75
    )

    &

    (
        test_results["prediction_proba"] <= 0.85
    )
]
test_results.sort_values(
    by="prediction_proba"
).head(50)

NameError: name 'test_results' is not defined

In [149]:
joblib.dump(

    model,

    "models/home_dnb_model.pkl"
)

['models/home_dnb_model.pkl']

In [150]:
joblib.dump(

    scaler,

    "models/home_dnb_scaler.pkl"
)

['models/home_dnb_scaler.pkl']

In [84]:
upcoming_matches = matches_features[

    (
        matches_features["season"] == 2025
    )

    &

    (
        matches_features["competition_name"]
        == "La Liga"
    )

    &

    (
        matches_features["status"] == "NS"
    )
].copy()

In [85]:
future_matches = matches_df[

    (
        matches_df["competition_name"]
        == "La Liga"
    )

    &

    (
        matches_df["status"] == "NS"
    )
].copy()

In [86]:
teams_query = """
SELECT
    id,
    name
FROM teams
"""

teams_df = pd.read_sql(
    teams_query,
    conn
)

team_map = dict(
    zip(
        teams_df["id"],
        teams_df["name"]
    )
)

future_matches = future_matches.merge(

    latest_home[
        [
            "home_rolling_scored",
            "home_rolling_conceded",
            "home_elo",
            "round_number"
        ]
    ],

    left_on="home_team_id",

    right_index=True,

    how="left"
)

future_matches = future_matches.merge(

    latest_away[
        [
            "away_rolling_scored",
            "away_rolling_conceded",
            "away_elo"
        ]
    ],

    left_on="away_team_id",

    right_index=True,

    how="left"
)

future_matches["home_team_name"] = (
    future_matches["home_team_id"]
    .map(team_map)
)

future_matches["away_team_name"] = (
    future_matches["away_team_id"]
    .map(team_map)
)

features = [

    "home_rolling_scored",

    "away_rolling_scored",

    "home_rolling_conceded",

    "away_rolling_conceded",

    "home_elo",

    "away_elo",

    "round_number"
]

X_future = future_matches[
    features
]

X_future_scaled = scaler.transform(
    X_future
)

probabilities = model.predict_proba(
    X_future_scaled
)

future_matches["home_dnb_proba"] = (
    probabilities[:, 1]
)

future_matches[
    [
        "date",

        "home_team_name",

        "away_team_name",

        "home_dnb_proba"
    ]
].sort_values(
    by="home_dnb_proba",
    ascending=False
)

,date,home_team_name,away_team_name,home_dnb_proba
20108,2026-05-23 19:00:00.000000,Real Madrid,Athletic Club,0.879978
20110,2026-05-23 19:00:00.000000,Real Betis,Levante,0.846406
20106,2026-05-23 19:00:00.000000,Celta Vigo,Sevilla,0.799734
20113,2026-05-23 19:00:00.000000,Mallorca,Oviedo,0.783196
20112,2026-05-23 19:00:00.000000,Girona,Elche,0.748870
20114,2026-05-24 19:00:00.000000,Villarreal,Atletico Madrid,0.741783
20111,2026-05-23 19:00:00.000000,Getafe,Osasuna,0.669390
20109,2026-05-23 19:00:00.000000,Alaves,Rayo Vallecano,0.653589
20107,2026-05-23 19:00:00.000000,Espanyol,Real Sociedad,0.595419
20105,2026-05-23 19:00:00.000000,Valencia,Barcelona,0.253489


In [53]:
X_future = future_matches[features]

probabilities = calibrated_model.predict_proba(
    X_future
)

KeyError: "None of [Index(['home_rolling_scored', 'away_rolling_scored', 'home_rolling_conceded',\n       'away_rolling_conceded', 'home_elo', 'away_elo', 'round_number'],\n      dtype='str')] are in the [columns]"

In [78]:
future_matches.columns.tolist()

['id',
 'api_fixture_id',
 'competition_id',
 'season',
 'date',
 'round',
 'home_team_id',
 'away_team_id',
 'home_goals',
 'away_goals',
 'status',
 'referee',
 'competition_name',
 'result',
 'total_goals',
 'over_2_5',
 'btts',
 'is_draw',
 'home_rolling_scored_x',
 'home_rolling_conceded_x',
 'home_elo_x',
 'round_number_x',
 'away_rolling_scored_x',
 'away_rolling_conceded_x',
 'away_elo_x',
 'home_dnb_proba',
 'home_rolling_scored_y',
 'home_rolling_conceded_y',
 'home_elo_y',
 'round_number_y',
 'away_rolling_scored_y',
 'away_rolling_conceded_y',
 'away_elo_y',
 'home_team_name',
 'away_team_name']

In [79]:
X_future = future_matches[
    features
]
X_future_scaled = scaler.transform(
    X_future
)
probabilities = calibrated_model.predict_proba(
    X_future_scaled
)
future_matches["home_dnb_proba"] = (
    probabilities[:, 1]
)
future_matches[
    [
        "date",

        "home_team_name",

        "away_team_name",

        "home_dnb_proba"
    ]
].sort_values(
    by="home_dnb_proba",
    ascending=False
)

KeyError: "None of [Index(['home_rolling_scored', 'away_rolling_scored', 'home_rolling_conceded',\n       'away_rolling_conceded', 'home_elo', 'away_elo', 'round_number'],\n      dtype='str')] are in the [columns]"

In [66]:
X_future = future_matches[
    features
]
X_future_scaled = scaler.transform(
    X_future
)
probabilities = model.predict_proba(
    X_future_scaled
)
future_matches["home_dnb_proba"] = (
    probabilities[:, 1]
)
future_matches[
    [
        "date",

        "home_team_name",

        "away_team_name",

        "home_dnb_proba"
    ]
].sort_values(
    by="home_dnb_proba",
    ascending=False
)

KeyError: "None of [Index(['home_rolling_scored', 'away_rolling_scored', 'home_rolling_conceded',\n       'away_rolling_conceded', 'home_elo', 'away_elo', 'round_number'],\n      dtype='str')] are in the [columns]"

NameError: name 'matches_features' is not defined